# 8. Neural networks that learn from the photos (run on Google Colab)

Notebooks 3 to 6 only saw 72 features we calculated by hand. These models look at the photos directly.

| Model | Why try it |
|---|---|
| MobileNetV3 | Very small and made for phones |
| EfficientNet-B0 | Small, usually a good balance |
| ConvNeXt-Tiny | Larger, shows how far a bigger model can go |

All three use the **same patients, same splits and same scores** as notebooks 3 to 7, so the results can be compared fairly.

**Before you start**

1. Open [colab.research.google.com](https://colab.research.google.com) and upload this notebook with **File > Upload notebook**.
2. Click **Runtime > Change runtime type**, choose **T4 GPU**, and save.
3. Click **Runtime > Run all**. Colab will ask to connect to your Google Drive, so that a dropped connection doesn't lose your work.

It downloads about 1.1 GB and may take 1 to 2 hours. If Colab disconnects, run it again and finished models are skipped.

In [ ]:
import subprocess

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Click Runtime > Change runtime type > T4 GPU, then run again.")
print("GPU:", torch.cuda.get_device_name(0))
print(subprocess.run("pip install -q openpyxl", shell=True, capture_output=True, text=True).stdout)

## Save to Google Drive

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = Path("/content/drive/MyDrive/AfriEye_neural_networks")
except ImportError:
    SAVE = Path("neural_network_output")
(SAVE / "results").mkdir(parents=True, exist_ok=True)
(SAVE / "models").mkdir(parents=True, exist_ok=True)
print("Saving to:", SAVE)

## Download the photos

In [ ]:
import os

REPO = Path("/content/DeepDRiD") if Path("/content").exists() else Path("DeepDRiD")
if not (REPO / "regular_fundus_images").exists():
    os.system(f"git clone --depth 1 --filter=blob:none --sparse https://github.com/deepdrdoc/DeepDRiD {REPO}")
    os.system(f"git -C {REPO} sparse-checkout set regular_fundus_images")
RAW = REPO / "regular_fundus_images"
print("Photos:", sum(1 for _ in RAW.rglob("*.jpg")))

## Load the labels

This uses the dataset's own patient-level split, the same as notebook 1. The checks stop the notebook if a patient appears in two splits.

In [ ]:
import json
import time

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import confusion_matrix, roc_auc_score


def load_split(name, split):
    folder = RAW / name
    d = pd.read_csv(folder / f"{name}.csv")
    d["grade"] = d["left_eye_DR_Level"].fillna(d["right_eye_DR_Level"])
    d = d.rename(columns={"Overall quality": "quality"})
    d["split"] = split
    d["path"] = [str(folder / "Images" / str(p) / f"{i}.jpg") for p, i in zip(d.patient_id, d.image_id)]
    return d[["patient_id", "image_id", "grade", "quality", "split", "path"]]


def load_test():
    folder = RAW / "Online-Challenge1&2-Evaluation"
    grades = pd.read_excel(folder / "Challenge1_labels.xlsx").rename(columns={"DR_Levels": "grade"})
    quality = pd.read_excel(folder / "Challenge2_labels.xlsx").rename(columns={"Overall quality": "quality"})
    d = grades.merge(quality[["image_id", "quality"]], on="image_id", how="left")
    d["patient_id"] = d.image_id.str.split("_").str[0].astype(int)
    d["split"] = "test"
    d["path"] = [str(folder / "Images" / str(p) / f"{i}.jpg") for p, i in zip(d.patient_id, d.image_id)]
    return d[["patient_id", "image_id", "grade", "quality", "split", "path"]]


data = pd.concat([load_split("regular-fundus-training", "train"), load_split("regular-fundus-validation", "val"), load_test()], ignore_index=True)
data = data[data.grade.isin([0, 1, 2, 3, 4])].reset_index(drop=True)
data["grade"] = data.grade.astype(int)
data["referable"] = (data.grade >= 2).astype(int)
assert data.split.value_counts().to_dict() == {"train": 1200, "val": 400, "test": 400}, data.split.value_counts().to_dict()
assert data.groupby("patient_id").split.nunique().max() == 1, "A patient appears in two splits."
assert all(Path(p).exists() for p in data.path), "Some photos are missing."
print(data.groupby("split").referable.agg(["count", "mean"]).round(3))

## Prepare the photos

Each photo is cropped to the retina and shrunk once, which makes training much faster.

**Image size is removed here on purpose.** Every photo becomes the same square, so the model can't read the original image size directly.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

SIZE = 320
CACHE = Path("/content/cache" if Path("/content").exists() else "cache") / str(SIZE)
CACHE.mkdir(parents=True, exist_ok=True)


def crop_to_retina(image, thumb=256, threshold=18):
    image = image.convert("RGB")
    w, h = image.size
    scale = thumb / max(w, h)
    small = np.asarray(image.resize((max(1, round(w * scale)), max(1, round(h * scale))), Image.BILINEAR))
    mask = small.max(axis=2) > threshold
    if mask.sum() >= 0.02 * mask.size:
        ys, xs = np.where(mask)
        image = image.crop((int(xs.min() / scale), int(ys.min() / scale), min(w, int((xs.max() + 1) / scale)), min(h, int((ys.max() + 1) / scale))))
    cw, ch = image.size
    side = max(cw, ch)
    canvas = Image.new("RGB", (side, side), (0, 0, 0))
    canvas.paste(image, ((side - cw) // 2, (side - ch) // 2))
    return canvas


def prepare(row):
    target = CACHE / f"{row.image_id}.jpg"
    if not target.exists():
        crop_to_retina(Image.open(row.path)).resize((SIZE, SIZE), Image.BILINEAR).save(target, quality=95)
    return str(target)


with ThreadPoolExecutor(4) as pool:
    data["small"] = list(pool.map(prepare, data.itertuples()))
print("Prepared", len(data), "photos")

## The training code

The model gives 5 answers for each photo: grade 1 or worse, grade 2 or worse (**referable**), grade 3 or worse (**urgent**), grade 4, and good photo quality. The app needs these for its result.

The best epoch is chosen with **validation data only**. The test photos aren't scored until a model has finished training.

In [ ]:
import torch.nn as nn
import torchvision as tv
from torch.utils.data import DataLoader, Dataset

MEAN, STD = [.485, .456, .406], [.229, .224, .225]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
normalize = tv.transforms.Normalize(MEAN, STD)
train_tf = tv.transforms.Compose([
    tv.transforms.RandomResizedCrop(SIZE, scale=(.85, 1.0), ratio=(.95, 1.05)), tv.transforms.RandomHorizontalFlip(), tv.transforms.RandomVerticalFlip(),
    tv.transforms.ColorJitter(.2, .2, .1, .02), tv.transforms.ToTensor(), normalize,
])
plain_tf = tv.transforms.Compose([tv.transforms.ToTensor(), normalize])


class Photos(Dataset):
    def __init__(self, frame, transform):
        self.d, self.t = frame.reset_index(drop=True), transform
    def __len__(self):
        return len(self.d)
    def __getitem__(self, i):
        r = self.d.iloc[i]
        y = torch.tensor([r.grade >= 1, r.grade >= 2, r.grade >= 3, r.grade >= 4, r.quality == 1], dtype=torch.float32)
        return self.t(Image.open(r.small).convert("RGB")), y


train, val, test = (data[data.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
loader = lambda frame, tf, shuffle: DataLoader(Photos(frame, tf), 24, shuffle=shuffle, num_workers=2, pin_memory=True, drop_last=shuffle)


def build(slug):
    if slug == "mobilenet_v3":
        m = tv.models.mobilenet_v3_large(weights="IMAGENET1K_V2"); m.classifier[3] = nn.Linear(m.classifier[3].in_features, 5)
    elif slug == "efficientnet_b0":
        m = tv.models.efficientnet_b0(weights="IMAGENET1K_V1"); m.classifier[1] = nn.Linear(m.classifier[1].in_features, 5)
    elif slug == "convnext_tiny":
        m = tv.models.convnext_tiny(weights="IMAGENET1K_V1"); m.classifier[2] = nn.Linear(m.classifier[2].in_features, 5)
    else:
        raise ValueError(slug)
    return m


@torch.no_grad()
def predict(model, frame):
    model.eval()
    out = []
    for x, _ in loader(frame, plain_tf, False):
        out.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy())
    return np.concatenate(out)


def fit(slug, epochs=18, lr=3e-4, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    model = build(slug).to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler("cuda")
    batches = loader(train, train_tf, True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(batches), pct_start=.1)
    best_auc, best_epoch, best_state, history = -1, 0, None, []
    for epoch in range(1, epochs + 1):
        model.train()
        total = 0
        for x, y in batches:
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = loss_fn(model(x.to(DEVICE)), y.to(DEVICE))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            total += loss.item()
        v = roc_auc_score(val.referable, predict(model, val)[:, 1])
        history.append({"epoch": epoch, "train_loss": total / len(batches), "val_auc": v})
        print(f"  epoch {epoch:2d}  loss {total / len(batches):.3f}  validation AUC {v:.3f}", flush=True)
        if v > best_auc:
            best_auc, best_epoch = v, epoch
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, history, best_epoch

## The scores

These are the same calculations as notebooks 3 to 6. The cut-off comes from validation photos only.

In [ ]:
def threshold_for_sensitivity(y, score, target=0.90):
    positives = np.sort(np.asarray(score)[np.asarray(y) == 1])[::-1]
    return float(positives[max(0, int(np.ceil(target * len(positives))) - 1)])


def rates(y, score, threshold):
    tn, fp, fn, tp = confusion_matrix(y, np.asarray(score) >= threshold, labels=[0, 1]).ravel()
    return {"sensitivity": float(tp / max(1, tp + fn)), "specificity": float(tn / max(1, tn + fp)), "tp": int(tp), "fn": int(fn), "tn": int(tn), "fp": int(fp)}


def patient_ci(metric, frame, score, repeats=1000, seed=0):
    rng = np.random.default_rng(seed)
    groups = frame.groupby("patient_id").indices
    ids = list(groups)
    y, score = frame.referable.values, np.asarray(score)
    results = []
    for _ in range(repeats):
        take = np.concatenate([groups[i] for i in rng.choice(ids, len(ids))])
        if 0 < y[take].sum() < len(take):
            results.append(metric(y[take], score[take]))
    return [float(np.percentile(results, 2.5)), float(np.percentile(results, 97.5))]


def report(name, val_p, test_p, seconds, extra):
    val_score, test_score = val_p[:, 1], test_p[:, 1]
    threshold = threshold_for_sensitivity(val.referable, val_score)
    v, t = rates(val.referable, val_score, threshold), rates(test.referable, test_score, threshold)
    result = {
        "model": name, "kind": "neural network", "seconds_to_train": seconds, "threshold": threshold,
        "urgent_threshold": threshold_for_sensitivity((val.grade >= 3).astype(int), val_p[:, 2], .80),
        "quality_threshold": float(np.percentile(val_p[:, 4], 15)),
        "val_auc": float(roc_auc_score(val.referable, val_score)), "val_sensitivity": v["sensitivity"], "val_specificity": v["specificity"],
        "test_auc": float(roc_auc_score(test.referable, test_score)), "test_auc_ci": patient_ci(roc_auc_score, test, test_score),
        "test_sensitivity": t["sensitivity"], "test_sensitivity_ci": patient_ci(lambda y, s: rates(y, s, threshold)["sensitivity"], test, test_score),
        "test_specificity": t["specificity"], "test_specificity_ci": patient_ci(lambda y, s: rates(y, s, threshold)["specificity"], test, test_score),
        "test_confusion": {k: t[k] for k in ("tp", "fn", "tn", "fp")},
        "missed_severe_or_worse": int(((test.grade >= 3) & (test_score < threshold)).sum()), "severe_or_worse": int((test.grade >= 3).sum()),
    }
    result.update(extra)
    return result


def speed_ms(model):
    model = model.cpu().eval()
    x = torch.zeros(1, 3, SIZE, SIZE)
    with torch.no_grad():
        model(x)
        start = time.time()
        for _ in range(10):
            model(x)
    return (time.time() - start) / 10 * 1000

## Train the three models

If a model is already saved in your Google Drive, it is skipped.

In [ ]:
MODELS_TO_TRAIN = {"mobilenet_v3": "MobileNetV3", "efficientnet_b0": "EfficientNet-B0", "convnext_tiny": "ConvNeXt-Tiny"}

for slug, name in MODELS_TO_TRAIN.items():
    done = SAVE / "results" / f"{slug}_metrics.json"
    if done.exists():
        print(name, "is already trained, skipping.")
        continue
    print("Training", name)
    start = time.time()
    model, history, best_epoch = fit(slug)
    seconds = time.time() - start
    val_p, test_p = predict(model, val), predict(model, test)
    weights = SAVE / "models" / f"{slug}.pt"
    torch.save(model.state_dict(), weights)
    result = report(name, val_p, test_p, seconds, {
        "best_epoch": best_epoch, "epochs": len(history), "image_size": SIZE, "parameters": int(sum(p.numel() for p in model.parameters())),
        "model_mb": round(weights.stat().st_size / 1e6, 1), "cpu_ms_per_photo": round(speed_ms(model), 1), "phone_ready": False,
    })
    frames = [f.assign(score=p[:, 1], urgent_score=p[:, 2], quality_score=p[:, 4]) for f, p in ((val, val_p), (test, test_p))]
    pd.concat(frames)[["image_id", "patient_id", "split", "grade", "referable", "score", "urgent_score", "quality_score"]].to_csv(SAVE / "results" / f"{slug}_predictions.csv", index=False)
    pd.DataFrame(history).to_csv(SAVE / "results" / f"{slug}_history.csv", index=False)
    done.write_text(json.dumps(result, indent=1))
    print(f"{name}: validation AUC {result['val_auc']:.3f}, test AUC {result['test_auc']:.3f}, severe missed {result['missed_severe_or_worse']} of {result['severe_or_worse']}\n")
    del model
    torch.cuda.empty_cache()

## The results

In [ ]:
rows = [json.loads(f.read_text()) for f in sorted((SAVE / "results").glob("*_metrics.json"))]
table = pd.DataFrame([{
    "Model": r["model"], "Val AUC": round(r["val_auc"], 3), "Test AUC": round(r["test_auc"], 3), "95% CI": f"{r['test_auc_ci'][0]:.2f} to {r['test_auc_ci'][1]:.2f}",
    "Sensitivity": f"{r['test_sensitivity']:.1%}", "Specificity": f"{r['test_specificity']:.1%}", "Severe missed": f"{r['missed_severe_or_worse']} of {r['severe_or_worse']}",
    "Size MB": r["model_mb"], "CPU ms": r["cpu_ms_per_photo"],
} for r in rows]).sort_values("Val AUC", ascending=False)
print(table.to_string(index=False))

## Download your results

This creates `afrieye_neural_networks.zip`. Unzip it into your **AfriEyeSamsung** folder, so that the files go into `results` and `models`. Then run notebooks 9 and 10 in VS Code.

In [ ]:
import shutil

archive = shutil.make_archive("afrieye_neural_networks", "zip", SAVE)
print("Created", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass